# Messy Dataset, Baseline Model

**MLSA SRM Technical Recruitment — AI/ML**

This notebook explores the deliberately messy recruitment-engagement dataset, documents the cleaning decisions, and compares two simple baseline classifiers. The goal is not to maximize accuracy, but to make every preprocessing and modeling choice explainable.

## 1. Problem statement

Predict whether an applicant completed the recruitment task (`completed_task`). The dataset contains missing values, inconsistent categorical representations, impossible numeric values, duplicate records, and several numerical outliers.

**Approach:** inspect first → clean only defensible data-quality problems → preserve plausible outliers → build reproducible preprocessing pipelines → compare Logistic Regression and Random Forest.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, ConfusionMatrixDisplay

pd.set_option('display.max_columns', None)

## 2. Load the starter dataset

The CSV is loaded directly from the official MLSA-SRM starter repository so the notebook is reproducible in Colab or locally with internet access.

In [ ]:
DATA_URL = 'https://raw.githubusercontent.com/MLSA-SRM/recruit-task-messy-dataset/main/recruitment_engagement.csv'
df = pd.read_csv(DATA_URL)

print('Shape:', df.shape)
display(df.head())
df.info()

## 3. Initial data-quality audit

Before changing anything, inspect types, missingness, unique categorical values, duplicates, and numerical ranges.

In [ ]:
print('Missing values:')
display(df.isna().sum().sort_values(ascending=False).to_frame('missing'))

print('Exact duplicate rows:', df.duplicated().sum())
print('Duplicate applicant IDs:', df['applicant_id'].duplicated().sum())

for col in ['domain', 'subdomain', 'year', 'signup_source', 'prior_experience', 'completed_task']:
    print(f'\n{col}:')
    display(df[col].value_counts(dropna=False).to_frame('count'))

display(df[['prep_hours_last_week', 'quiz_score', 'days_since_signup']].describe().T)

## 4. Cleaning decisions

### A. Categorical inconsistencies
- `domain`: normalize case (`technical`, `TECHNICAL`, `Technical` → `Technical`).
- `year`: map `First`, `1`, `1st Year` to `1st Year`; map `2`, `2nd year` to `2nd Year`.
- `prior_experience`: map `Y/yes/Yes` to `Yes` and `N/no/No` to `No`.
- `signup_source`: normalize capitalization.

### B. Impossible numeric values
- `prep_hours_last_week < 0` is impossible → treat as missing.
- `days_since_signup < 0` is impossible → treat as missing.
- `quiz_score` outside 0–100 would be impossible → treat as missing.

### C. Missing values
Do not drop rows simply because one field is missing. Numeric values are median-imputed and categorical values are most-frequent-imputed inside the modeling pipeline. This avoids throwing away applicants and prevents test-set information from leaking into training.

### D. Outliers
Large preparation-hour values are flagged with the IQR rule, but they are **not automatically deleted**. A high number of preparation hours is unusual, but not logically impossible. The Logistic Regression pipeline uses `RobustScaler`, while Random Forest is not scale-sensitive.

In [ ]:
clean = df.copy()

# Remove exact duplicate records only.
clean = clean.drop_duplicates().copy()

# Normalize categorical fields.
clean['domain'] = clean['domain'].astype('string').str.strip().str.lower().str.title()
clean['subdomain'] = clean['subdomain'].astype('string').str.strip()
clean['subdomain'] = clean['subdomain'].replace({'ai/ml': 'AI/ML', 'web dev': 'Web Dev'})
clean['subdomain'] = clean['subdomain'].str.title().replace({'Ai/Ml': 'AI/ML', 'Pr': 'PR'})
clean['year'] = clean['year'].astype('string').str.strip().str.lower().map({
    'first': '1st Year', '1': '1st Year', '1st year': '1st Year',
    'second': '2nd Year', '2': '2nd Year', '2nd year': '2nd Year'
})
clean['signup_source'] = clean['signup_source'].astype('string').str.strip().str.title()
clean['prior_experience'] = (
    clean['prior_experience'].astype('string').str.strip().str.lower()
.map({'yes': 'Yes', 'y': 'Yes', 'no': 'No', 'n': 'No'})
)

# Convert numeric columns explicitly.
numeric_cols = ['prep_hours_last_week', 'quiz_score', 'days_since_signup']
for col in numeric_cols:
    clean[col] = pd.to_numeric(clean[col], errors='coerce')

# Mark impossible values as missing.
clean.loc[clean['prep_hours_last_week'] < 0, 'prep_hours_last_week'] = np.nan
clean.loc[clean['days_since_signup'] < 0, 'days_since_signup'] = np.nan
clean.loc[~clean['quiz_score'].between(0, 100), 'quiz_score'] = np.nan

# Normalize target.
clean['completed_task'] = clean['completed_task'].astype('string').str.strip().str.title()

print('Rows before:', len(df))
print('Rows after exact-duplicate removal:', len(clean))
display(clean.head())

## 5. Outlier check

Use the IQR rule as a diagnostic rather than an automatic deletion rule. This lets us distinguish unusual values from impossible values.

In [ ]:
def iqr_outliers(series):
    s = series.dropna()
    q1, q3 = s.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return lower, upper, s[(s < lower) | (s > upper)]

for col in numeric_cols:
    lower, upper, out = iqr_outliers(clean[col])
    print(f'{col}: IQR bounds = ({lower:.2f}, {upper:.2f}); flagged = {len(out)}')
    if len(out):
        display(out.to_frame(col).sort_values(col))

## 6. Quick exploration

Look at the target balance and simple numeric relationships before modeling. These are descriptive checks, not evidence of causation.

In [ ]:
print('Target distribution:')
display(clean['completed_task'].value_counts(normalize=True).mul(100).round(1).to_frame('percent'))

display(clean.groupby('completed_task')[numeric_cols].median().round(2))

fig, ax = plt.subplots(figsize=(6, 4))
clean.boxplot(column='quiz_score', by='completed_task', ax=ax)
ax.set_title('Quiz score by task completion')
ax.set_xlabel('Completed task')
ax.set_ylabel('Quiz score')
plt.suptitle('')
plt.show()

## 7. Prepare features and split the data

`applicant_id` and `name` are excluded because they identify applicants rather than describe behavior. The split happens before fitting imputers/scalers/encoders so preprocessing is learned only from the training set.

In [ ]:
X = clean.drop(columns=['completed_task', 'applicant_id', 'name'])
y = clean['completed_task']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_features = X.select_dtypes(include=['object', 'string']).columns.tolist()
numeric_features = X.select_dtypes(include=['number']).columns.tolist()

print('Categorical:', categorical_features)
print('Numeric:', numeric_features)
print('Train:', X_train.shape, 'Test:', X_test.shape)

## 8. Preprocessing pipeline

Both models use the same preprocessing so the comparison is fair. Numeric missing values use the training median; categorical missing values use the training mode; categorical features are one-hot encoded. Logistic Regression additionally uses robust scaling because it is sensitive to feature scale and the dataset contains extreme numeric values.

In [ ]:
numeric_preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

categorical_preprocess = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_preprocess, numeric_features),
    ('cat', categorical_preprocess, categorical_features)
])

## 9. Baseline models

### Model 1 — Logistic Regression
A simple, interpretable linear baseline.

### Model 2 — Random Forest
A small nonlinear baseline that can capture interactions without requiring feature scaling. We keep it modest rather than tuning aggressively.

In [ ]:
logistic_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

rf_model = Pipeline([
    ('preprocess', preprocessor),
    ('model', RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced'))
])

models = {'Logistic Regression': logistic_model, 'Random Forest': rf_model}
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    results.append({
        'Model': name,
        'Accuracy': accuracy_score(y_test, pred),
        'Precision': precision_score(y_test, pred, pos_label='Yes'),
        'Recall': recall_score(y_test, pred, pos_label='Yes'),
        'F1': f1_score(y_test, pred, pos_label='Yes')
    })

results_df = pd.DataFrame(results).set_index('Model').round(3)
display(results_df)

## 10. Evaluation and error analysis

Accuracy alone is not enough. The confusion matrix and false positives/false negatives show what kinds of mistakes the model makes. Because this is a small dataset, the error analysis should be treated as a qualitative baseline rather than a final production conclusion.

In [ ]:
best_name = results_df['F1'].idxmax()
best_model = models[best_name]
best_pred = best_model.predict(X_test)

print('Selected by test F1 for inspection:', best_name)
print(classification_report(y_test, best_pred))

ConfusionMatrixDisplay.from_predictions(y_test, best_pred)
plt.title(f'Confusion Matrix — {best_name}')
plt.show()

errors = X_test.copy()
errors['actual'] = y_test
errors['predicted'] = best_pred
errors = errors[errors['actual'] != errors['predicted']].copy()
print('Misclassified test rows:', len(errors))
display(errors.sort_values(['actual', 'predicted']))

## 11. What I would ship

For this dataset, I would ship **the simpler model if its performance is close to the Random Forest**, because the dataset is tiny and the main value of this exercise is a reproducible, understandable baseline. If Random Forest is materially better on the held-out metrics, I would choose it while keeping the preprocessing pipeline unchanged.

I would not claim that this model is production-ready: the sample is small, the labels are recruitment-task outcomes, and a single train/test split can be noisy. A real system should use cross-validation, a larger dataset, and monitoring for changes in applicant behavior.

## 12. Key takeaways

1. The biggest issues are inconsistent representations, missing values, impossible negative values, duplicate records, and numerical outliers.
2. I corrected values only when there was a clear data-quality rule; unusual-but-possible values were retained.
3. Missing values are handled inside the pipeline to avoid leakage.
4. Two lightweight models are enough to establish a useful baseline without overengineering.
5. The model errors are more informative than accuracy alone, especially on this small dataset.